# 4 Train multilingual mBERT emotion classifier

This notebook builds the emotion classifier/evaluator used later to evalute whether Qwen's English translations preserve the original SMP emotion.

Qwen outputs are intentionally not used to train or fine-tune this model to avoid the dependency on the translation.

### Goal
Fine-tune `google-bert/bert-base-multilingual-cased` to predict:

- `joy`
- `anger`
- `sadness`
- `fear`
- `surprise`
- `neutral`

Training uses native text from both languages:

- **Chinese:** SMP2020-EWECT `usual` posts from `usual_splits.csv`
- **English:** GoEmotions Reddit comments


### Steps
1. Load `usual_splits.csv`
2. Create the six class labels `joy`, `anger`, `sadness`, `fear`, `surprise`, `neutral`
3. Load/filter GoEmotions to the same six class labels
4. Build Chinese + English train/validation/test sets
5. Create majority-class baseline model
6. Create a fine-tuned model with six-class head
7. Select the best checkpoint using validation macro-F1
8. Validate held-out Chinese and English performance
9. Freeze final calssifier

#### Outputs

- `models/mbert_emotion_6class/`
- `data/results/mbert_baseline_metrics.csv`
- `data/results/mbert_test_metrics.csv`
- `data/results/mbert_per_class_metrics.csv`
- `data/results/mbert_training_summary.json`

## 0. Install and import dependencies

In [1]:
%pip install -q -U "transformers>=4.45,<6" datasets accelerate scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

RANDOM_SEED = 266
set_seed(RANDOM_SEED)

#https://huggingface.co/google-bert/bert-base-multilingual-cased
MODEL_NAME = "google-bert/bert-base-multilingual-cased"
MAX_LENGTH = 128

current_folder = Path.cwd()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder

processed_folder = project_root / "data" / "processed"
results_folder = project_root / "data" / "results"
model_folder = project_root / "models" / "mbert_emotion_6class"


results_folder.mkdir(parents=True, exist_ok=True)
model_folder.mkdir(parents=True, exist_ok=True)

usual_splits_path = processed_folder / "usual_splits.csv"

print("Project root:", project_root)
print("Input:", usual_splits_path)
print("Model output:", model_folder)
print("Results output:", results_folder)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Project root: /content
Input: /content/data/processed/usual_splits.csv
Model output: /content/models/mbert_emotion_6class
Results output: /content/data/results
CUDA available: True
GPU: Tesla T4


## 1. Load SMP usual data

Note: SMP rows overlapping HADQAET must not be used to fine-tune mBERT because it will later act as human-annotated validation resource so we need to filter `split == "train" AND in_hadqaet == False`



In [7]:
print("Path:", usual_splits_path)
print("Exists:", usual_splits_path.exists())
print("Size:", usual_splits_path.stat().st_size, "bytes")

with open(usual_splits_path, "rb") as f:
    first_bytes = f.read(100)

print("First 100 raw bytes:")
print(first_bytes)

Path: /content/data/processed/usual_splits.csv
Exists: True
Size: 6811139 bytes
First 100 raw bytes:
b'\xef\xbb\xbfpost_id,split,text,label,in_hadqaet,mbert_train_eligible,has_slang_simple,slang_terms,slang_defin'


In [10]:
smp = pd.read_csv(usual_splits_path, encoding="utf-8-sig")

required_columns = {"post_id", "split", "text", "label", "in_hadqaet"}
missing = required_columns - set(smp.columns)

if missing:
  raise ValueError(f"Missing columns: {missing}")

print("Rows:", len(smp))
print("\nSplit counts:")
print(smp["split"].value_counts())
print("\nHADQAET-overlapping rows:", int(smp["in_hadqaet"].astype(bool).sum()))
print("mBERT-train-eligible rows:", int(smp["mbert_train_eligible"].astype(bool).sum()))

Rows: 34766

Split counts:
split
train    27864
eval      3457
test      3445
Name: count, dtype: int64

HADQAET-overlapping rows: 5977
mBERT-train-eligible rows: 21887


## 2. Create six label classes

GoEmotions dataset has 28 lables (27 emotions + neutral). We only need change 3 SMP labels (happy -> joy, angery -> anger, sad -> sadness).

| Shared label | SMP label | GoEmotions label |
|---|---|---|
| joy | happy | joy |
| anger | angry | anger |
| sadness | sad | sadness |
| fear | fear | fear |
| surprise | surprise | surprise |
| neutral | neutral | neutral |

In [13]:
LABELS = ["joy", "anger", "sadness", "fear", "surprise", "neutral"]

label_to_ids = {label: idx for idx, label in enumerate(LABELS)}
ids_to_label = {idx: label for label, idx in label_to_ids.items()}

# Map smp labels to shared labels
SMP_TO_SHARED = {
    "happy": "joy",
    "angry": "anger",
    "sad": "sadness",
    "fear": "fear",
    "surprise": "surprise",
    "neutral": "neutral"
}

smp["shared_label"] = smp["label"].map(SMP_TO_SHARED)

# Validate no unmapped
unmapped_smp = smp.loc[smp["shared_label"].isna(), "label"].unique().tolist()
if unmapped_smp:
    raise ValueError(f"Unmapped SMP labels found: {unmapped_smp}")

print("Shared labels:", LABELS)
print("Label ids:", label_to_ids)


Shared labels: ['joy', 'anger', 'sadness', 'fear', 'surprise', 'neutral']
Label ids: {'joy': 0, 'anger': 1, 'sadness': 2, 'fear': 3, 'surprise': 4, 'neutral': 5}


## 3. Prepare native Chinese SMPP train/validation/test sets
- Deduplicate exact text within each split
- Train only on `mbert_train_eligible == True`
- Use SMP `eval` as Chinese validation
- Keep SMP `test` held out for final classifier validation

**Note:** Notebook 2 already keeps duplicate text from crossing split boundaries

In [15]:
smp["normalized_text"] = smp["text"].astype(str).str.strip()

def deduplicate_text(df):
    return (
        df
        .drop_duplicates(subset=["normalized_text"], keep="first")
        .copy()
        .reset_index(drop=True)
    )

# Only take mBERT train eligible from the train split
train_raw = deduplicate_text(
    smp[smp["mbert_train_eligible"].astype(bool)]
)
val_raw = deduplicate_text(
    smp[smp["split"] == "eval"]
)
test_raw = deduplicate_text(
    smp[smp["split"] == "test"]
)

assert not train_raw["in_hadqaet"].astype(bool).any()

train_texts = set(train_raw["normalized_text"])
val_texts = set(val_raw["normalized_text"])
test_texts = set(test_raw["normalized_text"])

assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

print("Chinese train:", len(train_raw))
print("Chinese validation:", len(val_raw))
print("Chinese test:", len(test_raw))
print("\nChinese train label distribution:")
print(train_raw["shared_label"].value_counts())

Chinese train: 21375
Chinese validation: 3360
Chinese test: 3360

Chinese train label distribution:
shared_label
neutral     5702
anger       5604
joy         3997
sadness     3720
surprise    1491
fear         861
Name: count, dtype: int64


## 4. Load English GoEmotions data

From Gimini overview "The GoEmotions - Kaggle dataset features 58,000 curated Reddit comments labeled with 28 boolean emotion categories (27 fine-grained emotions plus Neutral). It is a multi-label dataset where a single comment can be tagged with one or multiple labels."

https://huggingface.co/datasets/google-research-datasets/go_emotions

In [16]:
go = load_dataset(
    "google-research-datasets/go_emotions",
    "simplified",
)

go_label_names = go["train"].features["labels"].feature.names

print("GoEmotions label count:", len(go_label_names))
print(go_label_names)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

GoEmotions label count: 28
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [17]:
def filter_goemotions(hf_split):
    rows = []
    for example in hf_split:
        label_names = [
            go_label_names[label_id]
            for label_id in example["labels"]
        ]

        if len(label_names) == 1 and label_names[0] in LABELS:
            rows.append({
                "text": example["text"],
                "shared_label": label_names[0],
            })

    return pd.DataFrame(rows)

en_train_raw = filter_goemotions(go["train"])
en_val_raw = filter_goemotions(go["validation"])
en_test_raw = filter_goemotions(go["test"])


print("English train:", len(en_train_raw))
print("English validation:", len(en_val_raw))
print("English test:", len(en_test_raw))
print("\nEnglish train label distribution:")
print(en_train_raw["shared_label"].value_counts())

English train: 16668
English validation: 2044
English test: 2084

English train label distribution:
shared_label
neutral     12823
anger        1025
joy           853
sadness       817
surprise      720
fear          430
Name: count, dtype: int64


## 5. Build multilingual train/validation/test sets

The mBERT training set combines native Chinese SMP posts and native English GoEmotions Reddit comments.

The validation set also combines both languages so checkpoint selection can reflect the multilingual classification task.

The test performance is reported separately for Chinese and English.

In [21]:
def to_classifier_frame(df, language, source):
  output_df = df[["text", "shared_label"]].copy()
  output_df["labels"] = output_df["shared_label"].map(label_to_ids)
  output_df["language"] = language
  output_df["source"] = source

  if output_df["labels"].isna().any():
        raise ValueError(f"Unmapped labels found in {source}")

  return output_df[["text", "shared_label", "labels", "language", "source"]]

In [24]:
# Get df from all Chinese and English sets
cn_train_df = to_classifier_frame(train_raw, "Chinese", "SMP")
cn_val_df = to_classifier_frame(val_raw, "Chinese", "SMP")
cn_test_df = to_classifier_frame(test_raw, "Chinese", "SMP")

en_train_df = to_classifier_frame(en_train_raw, "English", "GoEmotions")
en_val_df = to_classifier_frame(en_val_raw, "English", "GoEmotions")
en_test_df = to_classifier_frame(en_test_raw, "English", "GoEmotions")

# Combine Chinese and English train/validation/test sets,
# For each, shuffle the rows but set random_state to make sure it's reproducible
train_df = pd.concat([cn_train_df, en_train_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

val_df = pd.concat([cn_val_df, en_val_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

test_df = pd.concat([cn_test_df, en_test_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Total train:", len(train_df))
print("Total validation:", len(val_df))
print("Total test:", len(test_df))

print("\nTraining counts by language and emotion:")
display(pd.crosstab(train_df["shared_label"], train_df["language"]))

print("\nValidation counts by language and emotion:")
display(pd.crosstab(val_df["shared_label"], val_df["language"]))

Total train: 38043
Total validation: 5404
Total test: 5444

Training counts by language and emotion:


language,Chinese,English
shared_label,,
anger,5604,1025
fear,861,430
joy,3997,853
neutral,5702,12823
sadness,3720,817
surprise,1491,720



Validation counts by language and emotion:


language,Chinese,English
shared_label,,
anger,956,109
fear,145,58
joy,674,106
neutral,712,1592
sadness,619,84
surprise,254,95


## 6. Create majority-class classifier baseline
A simple baseline that predicts the most common training emotion

In [27]:
def majority_baseline(train_df, test_df, dataset_name):
  majority_id = int(train_df["labels"].value_counts().idxmax())

  y_true = test_df["labels"].to_numpy()
  y_pred = np.full(len(test_df), majority_id)

  return {
      "model": "majority_class",
      "dataset": dataset_name,
      "majority_label": ids_to_label[majority_id],
      "accuracy": accuracy_score(y_true, y_pred),
      "macro_f1": f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro"),
  }

In [29]:
baseline_metrics = pd.DataFrame([majority_baseline(cn_train_df, cn_test_df, "SMP Chinese Test"),
                                 majority_baseline(en_train_df, en_test_df, "GoEmotions English Test"),
                                 majority_baseline(train_df, test_df, "Combined Multilingual Test")])
baseline_path = results_folder / "mbert_baseline_metrics.csv"
baseline_metrics.to_csv(baseline_path, index=False)

display(baseline_metrics)
print("Saved:", baseline_path)

,model,dataset,majority_label,accuracy,macro_f1
0,majority_class,SMP Chinese Test,neutral,0.211905,0.058284
1,majority_class,GoEmotions English Test,neutral,0.770633,0.145077
2,majority_class,Combined Multilingual Test,neutral,0.425790,0.099545


Saved: /content/data/results/mbert_baseline_metrics.csv
